# Turni Employee Summary

This notebook reads the shared enriched files and writes the final employee summary for the configured root.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.notebooks.shared_config import load_notebook_context

ctx = load_notebook_context()
paths = ctx.paths
step_cfg = ctx.step("turni_employee_summary")

{
    "config_path": str(ctx.config_path),
    "input_dir": str(paths.enrichment_output),
    "output_dir": str(paths.aggregation_output),
}


In [ ]:
from src.drive_service.logging_utils import setup_logging
from src.turni_employee_summary.options import TurniEmployeeSummaryOptions

VERBOSE = True
report_path = paths.aggregation_output / step_cfg["report_name"]
summary_path = paths.aggregation_output / step_cfg["out_name"]
min_hours = step_cfg.get("min_hours")

options = TurniEmployeeSummaryOptions(
    enriched_dir=str(paths.enrichment_output),
    out=str(summary_path),
    report_json=str(report_path),
    output_format=str(step_cfg["output_format"]),
    year_start=int(step_cfg["year_start"]),
    year_end=int(step_cfg["year_end"]),
    min_hours=float(min_hours) if min_hours is not None else None,
    verbose=VERBOSE,
)
setup_logging(VERBOSE)
options


In [ ]:
from src.turni_employee_summary.service import run_from_options

report = run_from_options(options)

{
    "report_json": str(report_path),
    "output_path": report.get("output_path"),
    "rows": len(report.get("rows", [])),
}
